<a href="https://colab.research.google.com/github/electronicvan/BT4222_FinalProject_GoodreadsRecommender/blob/main/03_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook is used for Feature Engineering,

<h1> Baseline Feature Set </h1>

## Core IDs

- `user_id`  
  Unique identifier of the user associated with the event row.

- `book_id`  
  Unique identifier of the book associated with the event row.


## Temporal Tracking

- `event_time`  
  Primary event timestamp for the row, based on `date_added`, used to order interactions chronologically and enforce temporal feature construction.


## Static Item Metadata

- `title`  
  Raw book title.

- `description_truncated`  
  Truncated version of description, created to provide a more controlled text input for semantic encoding while limiting the influence of unusually long or noisy descriptions (cut off at 286 words)

- `top_shelf_tags`  
  Top cleaned informative shelf tags extracted from `popular_shelves` after excluding administrative, ownership, platform, and other non-content shelves.

- `has_meaningful_description`  
  Flag indicating whether the description passed the text-quality check and is considered sufficiently informative for downstream semantic use.

- `publication_year`  
  Publication year of the book.

- `num_pages`  
  Number of pages in the book.

- `language_code_collapsed`  
  Cleaned/collapsed language category for the book.

- `format_collapsed`  
  Cleaned/collapsed book format category.

- `is_ebook`  
  Indicator of whether the book is an ebook.

- `main_author_id`  
  Primary creator identifier used for author-based features. Preferably the first contributor with standardized role `author`; otherwise falls back to the first available contributor ID.

- `item_semantic_embedding_static`  
  Dense embedding from SentenceTransformer all-MiniLM-L6-v2, normalized.


## User-History / Matching features

- `num_pages_preference_gap`  
  Absolute difference between the current book's num_pages and the user's average num_pages of books in the user's prior interaction history before t. (user_page_preference_before_t)

- `user_author_interaction_share_before_t`
  Proportion of the user’s prior interactions before event time t, that involved the same main_author_id as the current book.

- `user_profile_embedding_similarity`
  Cosine similarity of the candidate book compared to the user’s prior semantic taste, quantified by user_profile_embedding_before_t


## Item Temporal Popularity

- `book_ratings_count_before_t`  
  Number of prior explicit ratings observed for the book before the current event time.

- `book_average_rating_before_t`  
  Average of prior explicit ratings observed for the book before the current event time.

## User Activity History

- `user_hist_interaction_count_before_t`
  Total number of interactions the user had before t. Basis for dense (>= 5) vs. sparse (< 5) user segmentation used in performance analysis.

- `user_hist_author_diversity_before_t`
  Count of distinct main_author_id values in the user's prior interaction history before t. Captures reading breadth.

- `days_since_user_last_interaction`
  Number of days between the last observed interaction of user and event time t.

<h1> Import Statements & Sample Rows of original dataframes </h1>

In [ ]:
import pandas as pd
from collections import Counter
import ast

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
books_df_cleaned = pd.read_csv("/content/drive/MyDrive/BT4222Project/CleanedDatasets/books_df_cleaned.csv")
reviews_df_cleaned = pd.read_csv("/content/drive/MyDrive/BT4222Project/CleanedDatasets/reviews_df_cleaned.csv")
interactions_df_cleaned = pd.read_csv("/content/drive/MyDrive/BT4222Project/CleanedDatasets/interactions_df_cleaned.csv")

In [ ]:
print("\nSample row from books_df_cleaned:")
display(books_df_cleaned.head(1).T)

print("\nSample row from interactions_df_cleaned:")
display(interactions_df_cleaned.head(1).T)

print("\nSample row from reviews_df_cleaned:")
display(reviews_df_cleaned.head(1).T)


Sample row from books_df_cleaned:


,0
isbn,1599150603
text_reviews_count,7
series,[]
country_code,US
language_code,NaN
popular_shelves,"[{'count': '56', 'name': 'to-read'}, {'count':..."
is_ebook,False
average_rating,4.13
similar_books,[]
description,"Relates in vigorous prose the tale of Aeneas, ..."



Sample row from interactions_df_cleaned:


,0
user_id,8842281e1d1347389f2ab93d60773d4d
book_id,10893214
review_id,5d0e4e8825c68740703f65a18813fc93
is_read,False
rating,0
date_added,2017-02-24 17:00:30+00:00
date_updated,2017-02-24 17:00:30+00:00
read_at,NaN
started_at,NaN
has_rating,0



Sample row from reviews_df_cleaned:


,0
user_id,8842281e1d1347389f2ab93d60773d4d
book_id,23310161
review_id,f4b4b050f4be00e9283c92a814af2670
rating,4
review_text,Fun sequel to the original.
date_added,2015-11-17 19:37:35+00:00
date_updated,2015-11-17 19:38:05+00:00
read_at,NaN
started_at,NaN
n_votes,7.0


<h1> Building Baseline Feature Set </h1>

In [ ]:
print(f"Cols in Books df: {books_df_cleaned.columns}")
print(f"Cols in Interactions df: {interactions_df_cleaned.columns}")
print(f"Cols in Reviews df: {reviews_df_cleaned.columns}")

Cols in Books df: Index(['isbn', 'text_reviews_count', 'series', 'country_code', 'language_code',
       'popular_shelves', 'is_ebook', 'average_rating', 'similar_books',
       'description', 'format', 'link', 'authors', 'publisher', 'num_pages',
       'publication_day', 'isbn13', 'publication_month', 'publication_year',
       'book_id', 'ratings_count', 'title', 'title_without_series',
       'language_code_missing', 'language_code_collapsed', 'format_collapsed',
       'authors_standardized', 'description_word_count',
       'has_meaningful_description', 'is_long_description',
       'description_truncated'],
      dtype='object')
Cols in Interactions df: Index(['user_id', 'book_id', 'review_id', 'is_read', 'rating', 'date_added',
       'date_updated', 'read_at', 'started_at', 'has_rating', 'cleaned_rating',
       'positive_rating'],
      dtype='object')
Cols in Reviews df: Index(['user_id', 'book_id', 'review_id', 'rating', 'review_text',
       'date_added', 'date_updated', '

In [ ]:
## Create static item table

items_static_cols = [
    "book_id",
    "title",
    "description",
    "description_truncated",
    "has_meaningful_description",
    "publication_year",
    "num_pages",
    "language_code_collapsed",
    "language_code_missing",
    "format_collapsed",
    "is_ebook",
    "authors_standardized",
    "popular_shelves",
    "ratings_count",
    "average_rating",
]

items_static_cols = [c for c in items_static_cols if c in books_df_cleaned.columns]

items_static_df = books_df_cleaned[items_static_cols].copy()

print(items_static_df.shape)
items_static_df.head()

(124037, 15)


,book_id,title,description,description_truncated,has_meaningful_description,publication_year,num_pages,language_code_collapsed,language_code_missing,format_collapsed,is_ebook,authors_standardized,popular_shelves,ratings_count,average_rating
0,287141,The Aeneid for Boys and Girls,"Relates in vigorous prose the tale of Aeneas, ...","Relates in vigorous prose the tale of Aeneas, ...",1,2006.0,162.0,unknown,1,paperback,False,"[{'author_id': '3041852', 'role_standardized':...","[{'count': '56', 'name': 'to-read'}, {'count':...",46,4.13
1,6066812,All's Fairy in Love and War (Avalon: Web of Ma...,"To Kara's astonishment, she discovers that a p...","To Kara's astonishment, she discovers that a p...",1,2009.0,216.0,unknown,1,paperback,False,"[{'author_id': '19158', 'role_standardized': '...","[{'count': '515', 'name': 'to-read'}, {'count'...",98,4.22
2,89378,Dog Heaven,In Newbery Medalist Cynthia Rylant's classic b...,In Newbery Medalist Cynthia Rylant's classic b...,1,1995.0,40.0,english,0,hardcover,False,"[{'author_id': '5411', 'role_standardized': 'a...","[{'count': '450', 'name': 'to-read'}, {'count'...",1331,4.43
3,3209312,"Moths and Mothers, Feathers and Fathers: A Sto...",NaN,NaN,0,NaN,NaN,unknown,1,unknown,False,"[{'author_id': '589328', 'role_standardized': ...","[{'count': '8', 'name': 'to-read'}, {'count': ...",11,4.29
4,1698376,What Do You Do?,WHAT DO YOU DO?\nA hen lays eggs...\nA cow giv...,WHAT DO YOU DO? A hen lays eggs... A cow gives...,1,2005.0,24.0,unknown,1,board_book,False,"[{'author_id': '169159', 'role_standardized': ...","[{'count': '8', 'name': 'to-read'}, {'count': ...",23,3.57


In [ ]:
## Create the event table from interactions

events_df = interactions_df_cleaned.copy()

# Use date_added as the primary event timestamp
events_df = events_df.rename(columns={"date_added": "event_time"})

event_cols = [
    "user_id",
    "book_id",
    "review_id",
    "event_time"
]

event_cols = [c for c in event_cols if c in events_df.columns]
events_df = events_df[event_cols].copy()

# Sort chronologically for later history features
events_df = events_df.sort_values(["user_id", "event_time", "book_id"]).reset_index(drop=True)

print(events_df.shape)
events_df.head()

(8770364, 4)


,user_id,book_id,review_id,event_time
0,00000377eea48021d3002730d56aca9a,13023,b5c7c0ca89779a65111ae924633ecf38,2014-12-07 20:11:06+00:00
1,00000377eea48021d3002730d56aca9a,41684,1d73829a06fe385376ecee3c29cb8be6,2014-12-07 20:13:38+00:00
2,00000377eea48021d3002730d56aca9a,34268,b521eeebe231b781e7176f35a8006cb4,2014-12-07 20:14:28+00:00
3,00000377eea48021d3002730d56aca9a,3636,fb806519c054c34a2c67b02ffc9fbb22,2014-12-07 20:15:18+00:00
4,00000377eea48021d3002730d56aca9a,152380,e8bc34e625167d545dc0ad2550e8aca9,2016-01-21 11:41:35+00:00


In [ ]:
## Merge static item features into events

baseline_df = events_df.merge(
    items_static_df,
    on="book_id",
    how="left",
    validate="many_to_one"
)

print(baseline_df.shape)
baseline_df.head()

(8770364, 18)


,user_id,book_id,review_id,event_time,title,description,description_truncated,has_meaningful_description,publication_year,num_pages,language_code_collapsed,language_code_missing,format_collapsed,is_ebook,authors_standardized,popular_shelves,ratings_count,average_rating
0,00000377eea48021d3002730d56aca9a,13023,b5c7c0ca89779a65111ae924633ecf38,2014-12-07 20:11:06+00:00,Alice in Wonderland,This is an adaptation. For the editions of the...,This is an adaptation. For the editions of the...,1,2004.0,92.0,english,0,hardcover,False,"[{'author_id': '34933', 'role_standardized': '...","[{'count': '140329', 'name': 'to-read'}, {'cou...",346530,4.03
1,00000377eea48021d3002730d56aca9a,41684,1d73829a06fe385376ecee3c29cb8be6,2014-12-07 20:13:38+00:00,The Jungle Books,The Jungle Books can be regarded as classic st...,The Jungle Books can be regarded as classic st...,1,2005.0,368.0,english,0,paperback,False,"[{'author_id': '6989', 'role_standardized': 'a...","[{'count': '1555', 'name': 'classics'}, {'coun...",69923,4.01
2,00000377eea48021d3002730d56aca9a,34268,b521eeebe231b781e7176f35a8006cb4,2014-12-07 20:14:28+00:00,Peter Pan,"Peter Pan, the book based on J.M. Barrie's fam...","Peter Pan, the book based on J.M. Barrie's fam...",1,NaN,NaN,english,0,unknown,False,"[{'author_id': '5255014', 'role_standardized':...","[{'count': '116674', 'name': 'to-read'}, {'cou...",173396,4.10
3,00000377eea48021d3002730d56aca9a,3636,fb806519c054c34a2c67b02ffc9fbb22,2014-12-07 20:15:18+00:00,"The Giver (The Giver, #1)",Twelve-year-old Jonas lives in a seemingly ide...,Twelve-year-old Jonas lives in a seemingly ide...,1,2006.0,208.0,english,0,paperback,False,"[{'author_id': '2493', 'role_standardized': 'a...","[{'count': '13366', 'name': 'to-read'}, {'coun...",1311422,4.12
4,00000377eea48021d3002730d56aca9a,152380,e8bc34e625167d545dc0ad2550e8aca9,2016-01-21 11:41:35+00:00,"Mary Poppins (Mary Poppins, #1)","By P.L. Travers, the author featured in the ma...","By P.L. Travers, the author featured in the ma...",1,2006.0,209.0,english,0,hardcover,False,"[{'author_id': '6872556', 'role_standardized':...","[{'count': '1938', 'name': 'to-read'}, {'count...",84109,4.05


In [ ]:
print(baseline_df.columns)

Index(['user_id', 'book_id', 'review_id', 'event_time', 'title', 'description',
       'description_truncated', 'has_meaningful_description',
       'publication_year', 'num_pages', 'language_code_collapsed',
       'language_code_missing', 'format_collapsed', 'is_ebook',
       'authors_standardized', 'popular_shelves', 'ratings_count',
       'average_rating'],
      dtype='object')


Item-Side Feature: Author
- main_author_id
- author_interactions_count_before_t



In [ ]:
# --------------------------------------------
# 1. Safely recreate main_author_id in books_df_cleaned
# --------------------------------------------
if "main_author_id" in books_df_cleaned.columns:
    books_df_cleaned = books_df_cleaned.drop(columns=["main_author_id"])

def get_main_author_id(authors_entry):
    if isinstance(authors_entry, str):
        try:
            authors_entry = ast.literal_eval(authors_entry)
        except Exception:
            return pd.NA

    if isinstance(authors_entry, dict):
        authors_entry = [authors_entry]

    if not isinstance(authors_entry, list):
        return pd.NA

    # First pass: prefer explicit author role
    for d in authors_entry:
        if isinstance(d, dict) and d.get("role_standardized") == "author":
            author_id = d.get("author_id")
            if pd.notna(author_id):
                return author_id

    # Fallback: first available author_id
    for d in authors_entry:
        if isinstance(d, dict):
            author_id = d.get("author_id")
            if pd.notna(author_id):
                return author_id

    return pd.NA

books_df_cleaned["main_author_id"] = books_df_cleaned["authors_standardized"].apply(
    get_main_author_id
)

# --------------------------------------------
# 2. Safely merge main_author_id into baseline_df
# --------------------------------------------
if "main_author_id" in baseline_df.columns:
    baseline_df = baseline_df.drop(columns=["main_author_id"])

baseline_df = baseline_df.merge(
    books_df_cleaned[["book_id", "main_author_id"]],
    on="book_id",
    how="left"
)

print(baseline_df[["book_id", "main_author_id"]].head(10))

num_valid = baseline_df["main_author_id"].notna().sum()
num_missing = baseline_df["main_author_id"].isna().sum()
total = len(baseline_df)

print(f"\nRows with valid main_author_id in baseline_df: {num_valid} ({num_valid / total:.2%})")
print(f"Rows with missing main_author_id in baseline_df: {num_missing} ({num_missing / total:.2%})")

    book_id main_author_id
0     13023           8164
1     41684           6989
2     34268        5255014
3      3636           2493
4    152380        6872556
5         5        1077326
6  27986676        2738670
7      3636           2493
8    357664          13663
9     38709           6569

Rows with valid main_author_id in baseline_df: 8770364 (100.00%)
Rows with missing main_author_id in baseline_df: 0 (0.00%)


In [ ]:
# --------------------------------------------
# Safely recreate author_interactions_count_before_t
# Count of prior interactions for the current row's main_author_id
# in strict event_time order. Starts at 0 for each author's
# first observed interaction.
# --------------------------------------------
if "author_interactions_count_before_t" in baseline_df.columns:
    baseline_df = baseline_df.drop(columns=["author_interactions_count_before_t"])

baseline_df = baseline_df.sort_values(
    ["event_time", "user_id", "book_id"]
).reset_index(drop=True)

baseline_df["author_interactions_count_before_t"] = (
    baseline_df.groupby("main_author_id").cumcount()
)

baseline_df.loc[
    baseline_df["main_author_id"].isna(),
    "author_interactions_count_before_t"
] = pd.NA

print(
    baseline_df[
        ["user_id", "book_id", "event_time", "main_author_id", "author_interactions_count_before_t"]
    ].head(20)
)

num_valid = baseline_df["author_interactions_count_before_t"].notna().sum()
num_missing = baseline_df["author_interactions_count_before_t"].isna().sum()
total = len(baseline_df)

print(f"\nRows with valid author_interactions_count_before_t: {num_valid} ({num_valid / total:.2%})")
print(f"Rows with missing author_interactions_count_before_t: {num_missing} ({num_missing / total:.2%})")

                             user_id  book_id                 event_time  \
0   dcdc48c7d14a6467cf277c9c0919eb02     6310  1990-01-01 08:00:00+00:00   
1   dcdc48c7d14a6467cf277c9c0919eb02     6319  1990-01-01 08:00:00+00:00   
2   dcdc48c7d14a6467cf277c9c0919eb02     6689  1990-01-01 08:00:00+00:00   
3   dcdc48c7d14a6467cf277c9c0919eb02    10365  1990-01-01 08:00:00+00:00   
4   dcdc48c7d14a6467cf277c9c0919eb02    39988  1990-01-01 08:00:00+00:00   
5   dcdc48c7d14a6467cf277c9c0919eb02   385247  1990-01-01 08:00:00+00:00   
6   dcdc48c7d14a6467cf277c9c0919eb02   108999  1992-01-01 08:00:00+00:00   
7   dcdc48c7d14a6467cf277c9c0919eb02   299085  1992-01-01 08:00:00+00:00   
8   dcdc48c7d14a6467cf277c9c0919eb02    24178  1992-03-29 08:00:00+00:00   
9   dcdc48c7d14a6467cf277c9c0919eb02  1248471  1995-01-01 08:00:00+00:00   
10  12707977a55df3ec3deaf86e874fe3ad   343114  1998-06-06 07:00:00+00:00   
11  12707977a55df3ec3deaf86e874fe3ad   125518  1998-09-01 07:00:00+00:00   
12  dd9db636

Item-Side Feature: Shelf Tags
- top_shelf_tags

In [ ]:
# --- Print unique shelf names from popular_shelves with counts and percentages ---
def parse_shelves_entry(entry):
    """Safely parse the popular_shelves entry into a list of dicts."""
    if entry is None:
        return []
    if isinstance(entry, list):
        return entry
    if isinstance(entry, dict):
        return [entry]
    if isinstance(entry, str):
        entry = entry.strip()
        if not entry:
            return []
        try:
            obj = ast.literal_eval(entry)
            if isinstance(obj, dict):
                return [obj]
            if isinstance(obj, list):
                return obj
        except Exception:
            pass
    return []

def get_all_shelf_names(entry):
    names = []
    parsed = parse_shelves_entry(entry)
    for d in parsed:
        if isinstance(d, dict) and 'name' in d:
            name = str(d['name']).strip().lower()
            names.append(name)
    return names

from collections import Counter
all_shelf_names = []
for entry in books_df_cleaned['popular_shelves']:
    all_shelf_names.extend(get_all_shelf_names(entry))

shelf_counter = Counter(all_shelf_names)
total_books = len(books_df_cleaned)
print(f"Unique shelf names found: {len(shelf_counter)}\n")
print(f"{'Shelf Name':<30} {'Count':>8} {'% of books':>12}")
for name, count in shelf_counter.most_common():
    pct = count / total_books * 100
    print(f"{name:<30} {count:>8} {pct:>11.2f}%")


Streaming output truncated to the last 5000 lines.
juv-pic-bk-i-loved-zander-didn-t        1        0.00%
forts                                 1        0.00%
toby-grade-one                        1        0.00%
authors-in-classroom-library          1        0.00%
future-k                              1        0.00%
religion-spirituality-magic           1        0.00%
escrito-en-espanol                    1        0.00%
for-my-family                         1        0.00%
dominican-republic-santo-domingo        1        0.00%
good-translations                     1        0.00%
possible-bb                           1        0.00%
environmental-activism                1        0.00%
read-alouds-spring                    1        0.00%
read-with-myla                        1        0.00%
2017_reading_challenge                1        0.00%
lexie-books                           1        0.00%
yoko-tanaka-illustrator               1        0.00%
let-s-add-all-books                   1     

In [ ]:
# --- Extract Top Shelf Tags ---
import ast
import re

# Obvious exclusions: administrative / status / ownership / platform shelves
OBVIOUS_SHELF_EXCLUSIONS = {
    # Reading status / workflow
    "to-read", "currently-reading", "read", "finished", "unread",
    "did-not-finish", "didn-t-finish", "abandoned", "not-read",
    "already-read", "reread", "re-read", "to-re-read", "to-reread",

    # Ownership / library / collection
    "owned", "books-i-own", "owned-books", "i-own", "own-it", "we-own",
    "books-we-own", "books-i-have", "my-books", "my-bookshelf", "bookshelf",
    "my-library", "library", "library-book", "library-books", "school-library",
    "public-library", "personal-library", "home-library", "our-library",
    "in-library", "from-library", "borrowed", "borrowed-from-library",
    "library-available", "on-my-shelf", "on-my-bookshelf",
    "in-the-collection", "collection", "inventory",

    # Purchase / acquisition
    "to-buy", "want-to-buy", "need-to-buy", "to-get", "buy", "bought",
    "want-to-own", "own-to-read", "owned-to-read",

    # Generic admin / vague
    "default", "books", "book", "shelved", "other", "general", "have",
    "mine", "new", "all-books",

    # Ratings / review tracking
    "review", "reviewed", "reviews", "for-review", "to-review",
    "5-stars", "4-stars", "3-stars", "2-stars", "5-star", "4-star", "3-star",

    # Platform / format / source
    "kindle", "ebook", "ebooks", "e-book", "e-books", "audio", "audiobook",
    "audiobooks", "audio-book", "audio-books", "audible", "digital",
    "on-kindle", "kindle-books",

    # User-relationship / sentiment shelves
    "favorites", "favorite", "favourites", "favourite", "my-favorites",
    "all-time-favorites", "all-time-favourites", "favorite-books",
    "favourite-books", "childhood", "nostalgia", "my-childhood",
    "read-as-a-kid", "read-as-a-child", "from-my-childhood",
    "childhood-favorites", "childhood-favourites", "childhood-faves",
    "childhood-memories", "books-from-my-childhood", "books-from-childhood",
    "when-i-was-young", "when-i-was-a-kid",

    # Numeric / junk labels
    "1", "2", "3", "a", "e", "x", "pb", "sub",
}

YEAR_SHELF_PATTERNS = [
    r"^read[-_ ]?in[-_ ]?\d{4}$",
    r"^read[-_ ]?\d{4}$",
    r"^\d{4}[-_ ]?reads$",
    r"^\d{4}[-_ ]?books$",
    r"^\d{4}[-_ ]?read$",
]

def is_excluded_shelf(name: str) -> bool:
    """Return True if the cleaned shelf name should be excluded."""
    if name in OBVIOUS_SHELF_EXCLUSIONS:
        return True
    for pattern in YEAR_SHELF_PATTERNS:
        if re.fullmatch(pattern, name):
            return True
    return False

def extract_top_shelf_tags(shelves_entry, top_n=5):
    """Extract top N informative shelf names after excluding obvious non-content shelves."""
    shelves = parse_shelves_entry(shelves_entry)
    shelf_list = []

    for d in shelves:
        if not isinstance(d, dict):
            continue

        name = d.get("name")
        count = d.get("count")

        if name is None:
            continue

        name_clean = str(name).strip().lower()
        if not name_clean or is_excluded_shelf(name_clean):
            continue

        try:
            count_num = int(count)
        except Exception:
            count_num = 0

        shelf_list.append((name_clean, count_num))

    # Sort by count descending, then alphabetically
    shelf_list.sort(key=lambda x: (-x[1], x[0]))

    # Deduplicate names while preserving order
    seen = set()
    top_names = []
    for name, _ in shelf_list:
        if name not in seen:
            seen.add(name)
            top_names.append(name)
        if len(top_names) == top_n:
            break

    return top_names

# 1. Create top_shelf_tags on books_df_cleaned
books_df_cleaned["top_shelf_tags"] = books_df_cleaned["popular_shelves"].apply(extract_top_shelf_tags)

# 2. Merge top_shelf_tags into baseline_df (not events_df)
if "top_shelf_tags" in baseline_df.columns:
    baseline_df = baseline_df.drop(columns=["top_shelf_tags"])

baseline_df = baseline_df.merge(
    books_df_cleaned[["book_id", "top_shelf_tags"]],
    on="book_id",
    how="left"
)

# 3. Print stats and sample from books_df_cleaned
num_nonempty = books_df_cleaned["top_shelf_tags"].apply(lambda x: bool(x)).sum()
total = len(books_df_cleaned)
print(f"Rows with non-empty top_shelf_tags in books_df_cleaned: {num_nonempty} ({num_nonempty / total:.2%})")

sample_cols = ["book_id", "title", "popular_shelves", "top_shelf_tags"]
sample = books_df_cleaned[sample_cols].head(10)
print("\nSample of top_shelf_tags from books_df_cleaned:")
print(sample.to_string(index=False))

# 4. Verify merge into baseline_df
num_baseline_nonempty = baseline_df["top_shelf_tags"].apply(lambda x: isinstance(x, list) and len(x) > 0).sum()
total_baseline = len(baseline_df)
print(f"\nRows with non-empty top_shelf_tags in baseline_df: {num_baseline_nonempty} ({num_baseline_nonempty / total_baseline:.2%})")

baseline_sample_cols = ["user_id", "book_id", "event_time", "top_shelf_tags"]
baseline_sample_cols = [c for c in baseline_sample_cols if c in baseline_df.columns]

print("\nSample of merged top_shelf_tags in baseline_df:")
print(baseline_df[baseline_sample_cols].head(10).to_string(index=False))

# 5. Print most common retained shelf tags after exclusion
from collections import Counter

all_retained_shelves = [
    shelf
    for shelf_list in books_df_cleaned["top_shelf_tags"]
    if isinstance(shelf_list, list)
    for shelf in shelf_list
]

shelf_counter = Counter(all_retained_shelves)

print("\nTop retained shelf tags after removing obvious non-content shelves:")
for shelf, count in shelf_counter.most_common(30):
    print(f"{shelf}: {count}")

Rows with non-empty top_shelf_tags in books_df_cleaned: 124037 (100.00%)

Sample of top_shelf_tags from books_df_cleaned:
 book_id                                                                         title                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        

Item-Side Semantic Embedding
- item_text_for_embedding (creating combined text string of title + meaningful description + top shelf tags)
- item_semantic_embedding_static
    - using SBERT (Sentence-BERT)


In [ ]:
# --- Create item_text_for_embedding ---

def build_item_text_for_embedding(row):
    parts = []

    # Title
    title = row.get("title", None)
    if pd.notna(title):
        title_str = str(title).strip()
        if title_str:
            parts.append(title_str)

    # Description: include only if meaningful, and use truncated version
    if int(row.get("has_meaningful_description", 0)) == 1:
        description = row.get("description_truncated", None)
        if pd.notna(description):
            description_str = str(description).strip()
            if description_str:
                parts.append(description_str)

    # Top shelf tags
    tags = row.get("top_shelf_tags", [])
    if isinstance(tags, list):
        tags_clean = [str(tag).strip() for tag in tags if pd.notna(tag) and str(tag).strip()]
        if tags_clean:
            parts.append(" ".join(tags_clean))

    # Always return a string
    return " [SEP] ".join(parts) if parts else ""

books_df_cleaned["item_text_for_embedding"] = books_df_cleaned.apply(
    build_item_text_for_embedding,
    axis=1
)

# --- Summary ---
nonempty_mask = books_df_cleaned["item_text_for_embedding"].str.strip() != ""
num_nonempty = int(nonempty_mask.sum())
total = len(books_df_cleaned)
avg_len = books_df_cleaned.loc[nonempty_mask, "item_text_for_embedding"].str.len().mean()

print(f"Rows with non-empty item_text_for_embedding: {num_nonempty} ({num_nonempty / total:.2%})")
print(
    f"Average character length (non-empty only): {avg_len:.2f}"
    if num_nonempty > 0 else
    "Average character length (non-empty only): N/A"
)

sample_cols = [
    "book_id",
    "title",
    "has_meaningful_description",
    "description_truncated",
    "top_shelf_tags",
    "item_text_for_embedding",
]
sample_cols = [c for c in sample_cols if c in books_df_cleaned.columns]

print("\nSample of item_text_for_embedding:")
print(books_df_cleaned[sample_cols].head(10).to_string(index=False))

Rows with non-empty item_text_for_embedding: 124037 (100.00%)
Average character length (non-empty only): 550.19

Sample of item_text_for_embedding:
 book_id                                                                         title  has_meaningful_description                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [ ]:
# --- Merge item_text_for_embedding into baseline_df and verify the merge ---

# Drop existing column first to avoid duplicate-column issues on re-run
if "item_text_for_embedding" in baseline_df.columns:
    baseline_df = baseline_df.drop(columns=["item_text_for_embedding"])

# Merge from books_df_cleaned by book_id
baseline_df = baseline_df.merge(
    books_df_cleaned[["book_id", "item_text_for_embedding"]],
    on="book_id",
    how="left",
    validate="many_to_one"
)

# --- Verification ---
num_nonempty = baseline_df["item_text_for_embedding"].fillna("").str.strip().ne("").sum()
total = len(baseline_df)

print(f"Rows with non-empty item_text_for_embedding in baseline_df: {num_nonempty} ({num_nonempty / total:.2%})")

verify_cols = ["user_id", "book_id", "event_time", "item_text_for_embedding"]
verify_cols = [c for c in verify_cols if c in baseline_df.columns]

print("\nSample of merged item_text_for_embedding in baseline_df:")
print(baseline_df[verify_cols].head(10).to_string(index=False))

Rows with non-empty item_text_for_embedding in baseline_df: 8770364 (100.00%)

Sample of merged item_text_for_embedding in baseline_df:
                         user_id  book_id                event_time                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [ ]:
# --- Safely recreate item_semantic_embedding_static in books_df_cleaned using SentenceTransformer ---

from sentence_transformers import SentenceTransformer
import numpy as np

# 1. Rerun-safe: drop existing embedding column if present
if "item_semantic_embedding_static" in books_df_cleaned.columns:
    books_df_cleaned = books_df_cleaned.drop(columns=["item_semantic_embedding_static"])

# 2. Load embedding model once
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# 3. Ensure text column is a clean string
books_df_cleaned["item_text_for_embedding"] = (
    books_df_cleaned["item_text_for_embedding"]
    .fillna("")
    .astype(str)
)

# 4. Encode text into dense semantic embeddings
item_embedding_matrix = embedding_model.encode(
    books_df_cleaned["item_text_for_embedding"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# 5. Store one embedding vector per row
books_df_cleaned["item_semantic_embedding_static"] = list(item_embedding_matrix)

# --- Summary / verification ---
num_nonempty = books_df_cleaned["item_text_for_embedding"].str.strip().ne("").sum()
total = len(books_df_cleaned)
embedding_dim = item_embedding_matrix.shape[1] if len(item_embedding_matrix) > 0 else 0

print(f"Rows with non-empty item_text_for_embedding: {num_nonempty} ({num_nonempty / total:.2%})")
print(f"Embedding dimension: {embedding_dim}")

sample_cols = [
    "book_id",
    "title",
    "item_text_for_embedding",
    "item_semantic_embedding_static",
]

print("\nSample of item_semantic_embedding_static in books_df_cleaned:")
print(
    books_df_cleaned[sample_cols]
    .head(5)
    .assign(
        item_semantic_embedding_static=lambda df: df["item_semantic_embedding_static"].apply(
            lambda x: np.array(x[:8]).round(4).tolist()
            if isinstance(x, (list, np.ndarray)) else x
        )
    )
    .to_string(index=False)
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1939 [00:00<?, ?it/s]

In [ ]:
# --- Merge item_semantic_embedding_static into baseline_df and verify the merge ---

import numpy as np

# Drop existing column first to make reruns safe
if "item_semantic_embedding_static" in baseline_df.columns:
    baseline_df = baseline_df.drop(columns=["item_semantic_embedding_static"])

baseline_df = baseline_df.merge(
    books_df_cleaned[["book_id", "item_semantic_embedding_static"]],
    on="book_id",
    how="left",
    validate="many_to_one"
)

# --- Verification ---
has_embedding = baseline_df["item_semantic_embedding_static"].apply(
    lambda x: isinstance(x, (list, np.ndarray)) and len(x) > 0
).sum()

total = len(baseline_df)
print(f"Rows with non-empty item_semantic_embedding_static in baseline_df: {has_embedding} ({has_embedding / total:.2%})")

verify_cols = ["user_id", "book_id", "event_time", "item_semantic_embedding_static"]
verify_cols = [c for c in verify_cols if c in baseline_df.columns]

print("\nSample of merged item_semantic_embedding_static in baseline_df:")
print(
    baseline_df[verify_cols]
    .head(10)
    .assign(
        item_semantic_embedding_static=lambda df: df["item_semantic_embedding_static"].apply(
            lambda x: np.array(x[:8]).round(4).tolist()
            if isinstance(x, (list, np.ndarray)) else x
        )
    )
    .to_string(index=False)
)

Rows with non-empty item_embedding_static in baseline_df: 6384165 (100.00%)

Sample of merged item_embedding_static in baseline_df:
                         user_id  book_id                event_time                                                                                                                                                               item_embedding_static
00000377eea48021d3002730d56aca9a    13023 2014-12-07 20:11:06+00:00   [0.09489999711513519, -0.002199999988079071, 0.005900000222027302, 0.08089999854564667, -0.006800000090152025, -0.007400000002235174, 0.09229999780654907, 0.0052999998442828655]
00000377eea48021d3002730d56aca9a    41684 2014-12-07 20:13:38+00:00 [-0.06589999794960022, 0.10779999941587448, -0.060600001364946365, 0.10499999672174454, -0.0031999999191612005, -0.014000000432133675, -0.020099999383091927, -0.08980000019073486]
00000377eea48021d3002730d56aca9a    34268 2014-12-07 20:14:28+00:00   [0.005900000222027302, -0.000699999975040555, 0.049800

User-Side Feature: User Behaviour before t

- num_pages_preference_gap - absolute difference between the current book’s num_pages and the user’s historical page preference before t
- user_hist_interaction_count_before_t
- days_since_user_last_interaction

In [ ]:
## num_pages_preference_gap

import numpy as np
import pandas as pd

# 0) Build lookup map from books_df_cleaned (no merge)
book_num_pages_map = books_df_cleaned.set_index("book_id")["num_pages"].to_dict()

# Safe rerun: drop/recreate helper + target feature columns on baseline_df
cols_to_drop = [
    "item_num_pages",
    "user_page_preference_before_t",
    "num_pages_preference_gap",
]
cols_to_drop = [c for c in cols_to_drop if c in baseline_df.columns]
if cols_to_drop:
    baseline_df = baseline_df.drop(columns=cols_to_drop)

# Add helper item attribute onto baseline_df via map
baseline_df["item_num_pages"] = baseline_df["book_id"].map(book_num_pages_map)

# 1) Ensure strict temporal ordering within user
baseline_df = baseline_df.sort_values(["user_id", "event_time", "book_id"]).reset_index(drop=True)

# 2) Initialize feature columns
baseline_df["user_page_preference_before_t"] = np.nan
baseline_df["num_pages_preference_gap"] = np.nan

# 3) Compute features user-by-user, row-by-row, using only prior rows
for user_id, idxs in baseline_df.groupby("user_id", sort=False).groups.items():
    prior_num_pages_sum = 0.0
    prior_num_pages_count = 0

    for idx in idxs:
        row = baseline_df.loc[idx]

        # Write current feature values using PRIOR state only
        if prior_num_pages_count > 0:
            user_page_pref = prior_num_pages_sum / prior_num_pages_count
            baseline_df.at[idx, "user_page_preference_before_t"] = user_page_pref

            curr_num_pages = row["item_num_pages"]
            if pd.notna(curr_num_pages):
                baseline_df.at[idx, "num_pages_preference_gap"] = abs(curr_num_pages - user_page_pref)

        # Update running state with CURRENT row after writing features
        curr_num_pages = row["item_num_pages"]
        if pd.notna(curr_num_pages):
            prior_num_pages_sum += curr_num_pages
            prior_num_pages_count += 1

# 4) Quick verification
feature_cols = [
    "user_id",
    "book_id",
    "event_time",
    "item_num_pages",
    "user_page_preference_before_t",
    "num_pages_preference_gap",
]
feature_cols = [c for c in feature_cols if c in baseline_df.columns]

print("\nSample of page-length preference features:")
print(baseline_df[feature_cols].head(20).to_string(index=False))


Sample of page-length preference features:
                         user_id  book_id                event_time  item_num_pages  is_read  user_page_preference_before_t  num_pages_preference_gap
00000377eea48021d3002730d56aca9a    13023 2014-12-07 20:11:06+00:00            92.0     True                            NaN                       NaN
00000377eea48021d3002730d56aca9a    41684 2014-12-07 20:13:38+00:00           368.0     True                      92.000000                276.000000
00000377eea48021d3002730d56aca9a    34268 2014-12-07 20:14:28+00:00             NaN     True                     230.000000                       NaN
00000377eea48021d3002730d56aca9a     3636 2014-12-07 20:15:18+00:00           208.0     True                     230.000000                 22.000000
00000377eea48021d3002730d56aca9a        5 2016-02-14 23:18:44+00:00           435.0     True                     222.666667                212.333333
00004584d524ec468619e81b176cc991     3636 2013-04-06 01:

In [ ]:
# --- Create user_hist_interaction_count_before_t on baseline_df (temporal, leakage-safe) ---

import pandas as pd
import numpy as np

# Safe rerun: drop existing column first
if "user_hist_interaction_count_before_t" in baseline_df.columns:
    baseline_df = baseline_df.drop(columns=["user_hist_interaction_count_before_t"])

# Ensure strict temporal ordering within user
baseline_df = baseline_df.sort_values(
    ["user_id", "event_time", "book_id"]
).reset_index(drop=True)

# Count prior interactions only
baseline_df["user_hist_interaction_count_before_t"] = (
    baseline_df.groupby("user_id").cumcount()
)

# Replace first interaction count 0 with NaN to exclude cold-start rows from segmentation
baseline_df["user_hist_interaction_count_before_t"] = baseline_df[
    "user_hist_interaction_count_before_t"
].replace(0, np.nan)

# --- Verification ---
verify_cols = [
    "user_id",
    "book_id",
    "event_time",
    "user_hist_interaction_count_before_t",
]
verify_cols = [c for c in verify_cols if c in baseline_df.columns]

print("\nSample of user_hist_interaction_count_before_t:")
print(baseline_df[verify_cols].head(20).to_string(index=False))

total = len(baseline_df)
num_missing = baseline_df["user_hist_interaction_count_before_t"].isna().sum()
num_sparse = (
    baseline_df["user_hist_interaction_count_before_t"].between(1, 4, inclusive="both")
).sum()
num_dense = (baseline_df["user_hist_interaction_count_before_t"] >= 5).sum()

print(f"\nRows with NaN (cold-start excluded): {num_missing} ({num_missing / total:.2%})")
print(f"Rows with sparse history (1-4): {num_sparse} ({num_sparse / total:.2%})")
print(f"Rows with dense history (>=5): {num_dense} ({num_dense / total:.2%})")

In [ ]:
# --- Create days_since_user_last_interaction on baseline_df (temporal, leakage-safe) ---

import numpy as np
import pandas as pd

# Safe rerun: drop existing column first
if "days_since_user_last_interaction" in baseline_df.columns:
    baseline_df = baseline_df.drop(columns=["days_since_user_last_interaction"])

# Ensure event_time is datetime
baseline_df["event_time"] = pd.to_datetime(baseline_df["event_time"], errors="coerce")

# Ensure strict temporal ordering within user
baseline_df = baseline_df.sort_values(
    ["user_id", "event_time", "book_id"]
).reset_index(drop=True)

# Previous interaction timestamp for the same user
baseline_df["prev_user_event_time"] = baseline_df.groupby("user_id")["event_time"].shift(1)

# Days since previous interaction
baseline_df["days_since_user_last_interaction"] = (
    baseline_df["event_time"] - baseline_df["prev_user_event_time"]
).dt.days

# Drop helper column to keep final table clean
baseline_df = baseline_df.drop(columns=["prev_user_event_time"])

# --- Verification ---
verify_cols = [
    "user_id",
    "book_id",
    "event_time",
    "days_since_user_last_interaction",
]
verify_cols = [c for c in verify_cols if c in baseline_df.columns]

print("\nSample of days_since_user_last_interaction:")
print(baseline_df[verify_cols].head(20).to_string(index=False))

total = len(baseline_df)
num_missing = baseline_df["days_since_user_last_interaction"].isna().sum()
num_nonmissing = baseline_df["days_since_user_last_interaction"].notna().sum()

print(f"\nRows with missing days_since_user_last_interaction: {num_missing} ({num_missing / total:.2%})")
print(f"Rows with non-missing days_since_user_last_interaction: {num_nonmissing} ({num_nonmissing / total:.2%})")

Item-Side Feature: Item Popularity
- book_interactions_count_before_t
- days_since_book_last_interaction

In [ ]:
# --- Create book_interactions_count_before_t on baseline_df (temporal, leakage-safe) ---

import pandas as pd

# Safe rerun: drop existing column first
if "book_interactions_count_before_t" in baseline_df.columns:
    baseline_df = baseline_df.drop(columns=["book_interactions_count_before_t"])

# Ensure strict temporal ordering across all rows
baseline_df = baseline_df.sort_values(
    ["event_time", "user_id", "book_id"]
).reset_index(drop=True)

# Count prior interactions for the same book only
baseline_df["book_interactions_count_before_t"] = (
    baseline_df.groupby("book_id").cumcount()
)

# --- Verification ---
verify_cols = [
    "user_id",
    "book_id",
    "event_time",
    "book_interactions_count_before_t",
]
verify_cols = [c for c in verify_cols if c in baseline_df.columns]

print("\nSample of book_interactions_count_before_t:")
print(baseline_df[verify_cols].head(20).to_string(index=False))

total = len(baseline_df)
num_zero = (baseline_df["book_interactions_count_before_t"] == 0).sum()
num_nonzero = (baseline_df["book_interactions_count_before_t"] > 0).sum()

print(f"\nRows with 0 prior book interactions: {num_zero} ({num_zero / total:.2%})")
print(f"Rows with >0 prior book interactions: {num_nonzero} ({num_nonzero / total:.2%})")

In [ ]:
# --- Create days_since_book_last_interaction on baseline_df (temporal, leakage-safe) ---

import pandas as pd

# Safe rerun: drop existing feature/helper columns first
cols_to_drop = [
    c for c in ["prev_book_event_time", "days_since_book_last_interaction"]
    if c in baseline_df.columns
]
if cols_to_drop:
    baseline_df = baseline_df.drop(columns=cols_to_drop)

# Ensure event_time is datetime
baseline_df["event_time"] = pd.to_datetime(baseline_df["event_time"], errors="coerce")

# Ensure strict temporal ordering across all rows
baseline_df = baseline_df.sort_values(
    ["event_time", "user_id", "book_id"]
).reset_index(drop=True)

# Previous interaction timestamp for the same book
baseline_df["prev_book_event_time"] = baseline_df.groupby("book_id")["event_time"].shift(1)

# Days since previous interaction of the same book
baseline_df["days_since_book_last_interaction"] = (
    baseline_df["event_time"] - baseline_df["prev_book_event_time"]
).dt.days

# Drop helper column to keep final table clean
baseline_df = baseline_df.drop(columns=["prev_book_event_time"])

# --- Verification ---
verify_cols = [
    "user_id",
    "book_id",
    "event_time",
    "days_since_book_last_interaction",
]
verify_cols = [c for c in verify_cols if c in baseline_df.columns]

print("\nSample of days_since_book_last_interaction:")
print(baseline_df[verify_cols].head(20).to_string(index=False))

total = len(baseline_df)
num_missing = baseline_df["days_since_book_last_interaction"].isna().sum()
num_nonmissing = baseline_df["days_since_book_last_interaction"].notna().sum()

print(f"\nRows with missing days_since_book_last_interaction: {num_missing} ({num_missing / total:.2%})")
print(f"Rows with non-missing days_since_book_last_interaction: {num_nonmissing} ({num_nonmissing / total:.2%})")

In [ ]:
print(f"Cols in baseline_df: {baseline_df.columns}")

Cols in baseline_df: Index(['user_id', 'book_id', 'review_id', 'event_time', 'cleaned_rating',
       'positive_rating', 'is_read', 'title', 'description',
       'has_meaningful_description', 'publication_year', 'num_pages',
       'language_code_collapsed', 'language_code_missing', 'format_collapsed',
       'is_ebook', 'authors_standardized', 'popular_shelves', 'ratings_count',
       'text_reviews_count', 'average_rating', 'main_author_id',
       'has_read_author_before_t', 'has_liked_author_before_t',
       'top_shelf_tags', 'item_text_for_embedding', 'item_embedding_static',
       'item_num_pages', 'user_page_preference_before_t',
       'num_pages_preference_gap', 'book_ratings_count_before_t',
       'book_average_rating_before_t', 'description_truncated'],
      dtype='object')


User-Side Feature: Author Affinity

- user_author_interaction_share_before_t
- user_hist_author_diversity_before_t

In [ ]:
# --- Create user_author_interaction_share_before_t on baseline_df (temporal, leakage-safe) ---

import numpy as np
import pandas as pd

# Safe rerun: drop existing column first
if "user_author_interaction_share_before_t" in baseline_df.columns:
    baseline_df = baseline_df.drop(columns=["user_author_interaction_share_before_t"])

# Ensure strict temporal ordering within user
baseline_df = baseline_df.sort_values(
    ["user_id", "event_time", "book_id"]
).reset_index(drop=True)

# Initialize feature column
baseline_df["user_author_interaction_share_before_t"] = np.nan

# Compute feature user-by-user using only prior rows
for user_id, idxs in baseline_df.groupby("user_id", sort=False).groups.items():
    total_prior_interactions = 0
    prior_author_counts = {}

    for idx in idxs:
        row = baseline_df.loc[idx]
        author_id = row["main_author_id"]

        # Write current row feature using PRIOR state only
        if total_prior_interactions > 0 and pd.notna(author_id):
            author_prior_count = prior_author_counts.get(author_id, 0)
            baseline_df.at[idx, "user_author_interaction_share_before_t"] = (
                author_prior_count / total_prior_interactions
            )

        # Update running state with CURRENT row after writing features
        if pd.notna(author_id):
            prior_author_counts[author_id] = prior_author_counts.get(author_id, 0) + 1

        total_prior_interactions += 1

# --- Verification ---
verify_cols = [
    "user_id",
    "book_id",
    "event_time",
    "main_author_id",
    "user_author_interaction_share_before_t",
]
verify_cols = [c for c in verify_cols if c in baseline_df.columns]

print("\nSample of user_author_interaction_share_before_t:")
print(baseline_df[verify_cols].head(20).to_string(index=False))

total = len(baseline_df)
num_missing = baseline_df["user_author_interaction_share_before_t"].isna().sum()
num_nonmissing = baseline_df["user_author_interaction_share_before_t"].notna().sum()

print(f"\nRows with missing user_author_interaction_share_before_t: {num_missing} ({num_missing / total:.2%})")
print(f"Rows with non-missing user_author_interaction_share_before_t: {num_nonmissing} ({num_nonmissing / total:.2%})")

In [ ]:
# --- Create user_hist_author_diversity_before_t on baseline_df (temporal, leakage-safe) ---

import pandas as pd
import numpy as np

# Safe rerun: drop existing column first
if "user_hist_author_diversity_before_t" in baseline_df.columns:
    baseline_df = baseline_df.drop(columns=["user_hist_author_diversity_before_t"])

# Ensure strict temporal ordering within user
baseline_df = baseline_df.sort_values(
    ["user_id", "event_time", "book_id"]
).reset_index(drop=True)

# Initialize feature column
baseline_df["user_hist_author_diversity_before_t"] = 0

# Compute feature user-by-user using only prior rows
for user_id, idxs in baseline_df.groupby("user_id", sort=False).groups.items():
    seen_authors = set()

    for idx in idxs:
        row = baseline_df.loc[idx]
        author_id = row["main_author_id"]

        # Write current row feature using PRIOR state only
        baseline_df.at[idx, "user_hist_author_diversity_before_t"] = len(seen_authors)

        # Update running state with CURRENT row after writing features
        if pd.notna(author_id):
            seen_authors.add(author_id)

# --- Verification ---
verify_cols = [
    "user_id",
    "book_id",
    "event_time",
    "main_author_id",
    "user_hist_author_diversity_before_t",
]
verify_cols = [c for c in verify_cols if c in baseline_df.columns]

print("\nSample of user_hist_author_diversity_before_t:")
print(baseline_df[verify_cols].head(20).to_string(index=False))

total = len(baseline_df)
num_zero = (baseline_df["user_hist_author_diversity_before_t"] == 0).sum()
num_positive = (baseline_df["user_hist_author_diversity_before_t"] > 0).sum()

print(f"\nRows with 0 prior author diversity: {num_zero} ({num_zero / total:.2%})")
print(f"Rows with >0 prior author diversity: {num_positive} ({num_positive / total:.2%})")

User Embedding
- user_profile_embedding_before_t
- user_profile_embedding_similarity

In [ ]:
# --- Create user_profile_embedding_before_t on baseline_df (temporal, leakage-safe) ---

import numpy as np
import pandas as pd

# Safe rerun: drop existing column first
if "user_profile_embedding_before_t" in baseline_df.columns:
    baseline_df = baseline_df.drop(columns=["user_profile_embedding_before_t"])

# Ensure strict temporal ordering within user
baseline_df = baseline_df.sort_values(
    ["user_id", "event_time", "book_id"]
).reset_index(drop=True)

# Find embedding dimension from the first valid item embedding
valid_item_embedding = next(
    (
        x for x in baseline_df["item_semantic_embedding_static"]
        if isinstance(x, (list, np.ndarray)) and len(x) > 0
    ),
    None
)

if valid_item_embedding is None:
    raise ValueError("No valid item_semantic_embedding_static found in baseline_df.")

embedding_dim = len(valid_item_embedding)

# Initialize output column
baseline_df["user_profile_embedding_before_t"] = pd.Series(
    [pd.NA] * len(baseline_df),
    dtype="object"
)

# Compute user profile embedding from prior interacted-item embeddings only
for user_id, idxs in baseline_df.groupby("user_id", sort=False).groups.items():
    running_sum = np.zeros(embedding_dim, dtype=float)
    running_count = 0

    for idx in idxs:
        row = baseline_df.loc[idx]
        curr_item_emb = row["item_semantic_embedding_static"]

        # Write current-row feature using PRIOR state only
        if running_count > 0:
            user_profile = running_sum / running_count

            # L2-normalize for consistency
            norm = np.linalg.norm(user_profile)
            if norm > 0:
                user_profile = user_profile / norm

            baseline_df.at[idx, "user_profile_embedding_before_t"] = user_profile

        # Update running state with CURRENT row after writing feature
        if isinstance(curr_item_emb, (list, np.ndarray)) and len(curr_item_emb) == embedding_dim:
            curr_item_emb = np.asarray(curr_item_emb, dtype=float)
            running_sum += curr_item_emb
            running_count += 1

# --- Verification ---
verify_cols = [
    "user_id",
    "book_id",
    "event_time",
    "user_profile_embedding_before_t",
]
verify_cols = [c for c in verify_cols if c in baseline_df.columns]

print("\nSample of user_profile_embedding_before_t:")
print(
    baseline_df[verify_cols]
    .head(10)
    .assign(
        user_profile_embedding_before_t=lambda df: df["user_profile_embedding_before_t"].apply(
            lambda x: np.array(x[:8]).round(4).tolist()
            if isinstance(x, (list, np.ndarray)) else x
        )
    )
    .to_string(index=False)
)

total = len(baseline_df)
num_missing = baseline_df["user_profile_embedding_before_t"].isna().sum()
num_nonmissing = baseline_df["user_profile_embedding_before_t"].notna().sum()

print(f"\nRows with missing user_profile_embedding_before_t: {num_missing} ({num_missing / total:.2%})")
print(f"Rows with non-missing user_profile_embedding_before_t: {num_nonmissing} ({num_nonmissing / total:.2%})")
print(f"Embedding dimension: {embedding_dim}")

In [ ]:
# --- Create user_profile_embedding_similarity on baseline_df ---

import numpy as np
import pandas as pd

# Safe rerun: drop existing column first
if "user_profile_embedding_similarity" in baseline_df.columns:
    baseline_df = baseline_df.drop(columns=["user_profile_embedding_similarity"])

def cosine_similarity_safe(a, b):
    if not isinstance(a, (list, np.ndarray)) or not isinstance(b, (list, np.ndarray)):
        return np.nan

    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    if a.size == 0 or b.size == 0 or a.shape != b.shape:
        return np.nan

    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)

    if norm_a == 0 or norm_b == 0:
        return np.nan

    return float(np.dot(a, b) / (norm_a * norm_b))

baseline_df["user_profile_embedding_similarity"] = baseline_df.apply(
    lambda row: cosine_similarity_safe(
        row["user_profile_embedding_before_t"],
        row["item_semantic_embedding_static"]
    ),
    axis=1
)

# --- Verification ---
verify_cols = [
    "user_id",
    "book_id",
    "event_time",
    "user_profile_embedding_similarity",
]
verify_cols = [c for c in verify_cols if c in baseline_df.columns]

print("\nSample of user_profile_embedding_similarity:")
print(baseline_df[verify_cols].head(20).to_string(index=False))

total = len(baseline_df)
num_missing = baseline_df["user_profile_embedding_similarity"].isna().sum()
num_nonmissing = baseline_df["user_profile_embedding_similarity"].notna().sum()

print(f"\nRows with missing user_profile_embedding_similarity: {num_missing} ({num_missing / total:.2%})")
print(f"Rows with non-missing user_profile_embedding_similarity: {num_nonmissing} ({num_nonmissing / total:.2%})")

if num_nonmissing > 0:
    print("\nSummary stats for user_profile_embedding_similarity:")
    print(baseline_df["user_profile_embedding_similarity"].describe())

<h1> Dropping Helper Columns </h1?

In [ ]:
# --- Keep only the final selected columns in baseline_df ---

# Columns to retain
cols_to_keep = [
    # Core IDs
    "user_id",
    "book_id",

    # Temporal tracking
    "event_time",

    # Static item metadata
    "title",
    "description_truncated",
    "top_shelf_tags",
    "has_meaningful_description",
    "publication_year",
    "num_pages",
    "language_code_collapsed",
    "format_collapsed",
    "is_ebook",
    "main_author_id",
    "item_semantic_embedding_static",

    # User-history / matching features
    "num_pages_preference_gap",
    "user_author_interaction_share_before_t",
    "user_profile_embedding_similarity",

    # Item temporal popularity
    "book_ratings_count_before_t",
    "book_average_rating_before_t",

    # User activity history
    "user_hist_interaction_count_before_t",
    "user_hist_author_diversity_before_t",
    "days_since_user_last_interaction",
]

# Keep only columns that actually exist in baseline_df
existing_cols_to_keep = [c for c in cols_to_keep if c in baseline_df.columns]

# Optional: show missing expected columns before subsetting
missing_cols = [c for c in cols_to_keep if c not in baseline_df.columns]
if missing_cols:
    print("These expected columns were not found in baseline_df and will be skipped:")
    for c in missing_cols:
        print(f"- {c}")

# Subset baseline_df to only the selected columns
baseline_df = baseline_df[existing_cols_to_keep].copy()

In [ ]:
print(f"Cols in baseline_df: {baseline_df.columns}")

<h1> Exporting </h1?

In [ ]:
baseline_df.to_pickle("baseline_df.pkl")
print("Exported baseline_df.pkl")

Exported baseline_df.pkl
